<a href="https://colab.research.google.com/github/Yascaram/Datathon-Fase-5-/blob/main/Limpeza_dos_dados_tech5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# %load limpeza_passos_magicos.py
import pandas as pd
import numpy as np
import re

# ── 1. Carregar os dados ──────────────────────────────────────────────────────
df22 = pd.read_excel("2022.xlsx")
df23 = pd.read_excel("2023.xlsx")
df24 = pd.read_excel("2024.xlsx")


# ── 2. Funções de limpeza por ano ─────────────────────────────────────────────

def clean_2022(df):
    d = df.copy()

    # Renomear colunas para padrão comum
    d.rename(columns={
        'Idade 22':   'Idade',
        'INDE 22':    'INDE',
        'Pedra 22':   'Pedra',
        'Matem':      'Mat',
        'Portug':     'Por',
        'Inglês':     'Ing',
        'Defas':      'Defasagem',
        'Fase ideal': 'Fase Ideal',
    }, inplace=True)

    d['Ano'] = 2022

    # Padronizar Gênero
    d['Gênero'] = d['Gênero'].map({'Menina': 'Feminino', 'Menino': 'Masculino'})

    # Padronizar Fase: numérica → string
    fase_map = {
        0: 'ALFA',   1: 'FASE 1', 2: 'FASE 2', 3: 'FASE 3',
        4: 'FASE 4', 5: 'FASE 5', 6: 'FASE 6', 7: 'FASE 7', 8: 'FASE 8',
    }
    d['Fase'] = d['Fase'].map(fase_map)

    # Padronizar Instituição
    d['Instituição de ensino'] = d['Instituição de ensino'].replace({
        'Escola Pública': 'Pública',
        'Rede Decisão':   'Privada',
        'Escola JP II':   'Privada',
    })

    return d


def clean_2023(df):
    d = df.copy()

    d.rename(columns={
        'Nome Anonimizado': 'Nome',
        'INDE 2023':        'INDE',
        'Pedra 2023':       'Pedra',
    }, inplace=True)

    d['Ano'] = 2023

    # Padronizar Fase (já é texto, só garantir maiúsculas e trim)
    d['Fase'] = d['Fase'].str.upper().str.strip()

    # Corrigir acentuação da Pedra
    d['Pedra'] = d['Pedra'].replace({'Agata': 'Ágata'})

    return d


def clean_2024(df):
    d = df.copy()

    d.rename(columns={
        'Nome Anonimizado': 'Nome',
        'INDE 2024':        'INDE',
        'Pedra 2024':       'Pedra',
    }, inplace=True)

    d['Ano'] = 2024

    # INDE veio como object — converter para numérico
    d['INDE'] = pd.to_numeric(d['INDE'], errors='coerce')

    # Fase: turmas como "3B", "7E" → "FASE 3", "FASE 7"; "ALFA" permanece
    def normalizar_fase(f):
        f = str(f).strip().upper()
        if f == 'ALFA':
            return 'ALFA'
        m = re.match(r'^(\d+)', f)           # extrai número inicial
        if m:
            n = int(m.group(1))
            if 1 <= n <= 8:
                return f'FASE {n}'
        return f                              # retorna original se não reconhecido

    d['Fase'] = d['Fase'].apply(normalizar_fase)

    # Corrigir Pedra
    d['Pedra'] = d['Pedra'].replace({'Agata': 'Ágata', 'INCLUIR': np.nan})

    return d


# ── 3. Aplicar limpezas ───────────────────────────────────────────────────────
df22c = clean_2022(df22)
df23c = clean_2023(df23)
df24c = clean_2024(df24)


# ── 4. Selecionar colunas comuns e unificar ───────────────────────────────────
COLS = [
    'Ano', 'RA', 'Nome', 'Fase', 'Fase Ideal', 'Defasagem',
    'Gênero', 'Idade', 'Ano ingresso', 'Instituição de ensino',
    'Pedra', 'INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPV', 'IAN',
    'Mat', 'Por', 'Ing',
]

def selecionar(df):
    return df[[c for c in COLS if c in df.columns]].copy()

base = pd.concat(
    [selecionar(df22c), selecionar(df23c), selecionar(df24c)],
    ignore_index=True,
)


# ── 5. Limpeza final na base unificada ────────────────────────────────────────

# Simplificar categorias de Instituição
base['Instituição de ensino'] = base['Instituição de ensino'].replace({
    'Privada - Programa de Apadrinhamento':     'Privada (Bolsa)',
    'Privada - Programa de apadrinhamento':     'Privada (Bolsa)',
    'Privada *Parcerias com Bolsa 100%':        'Privada (Bolsa)',
    'Privada - Pagamento por *Empresa Parceira':'Privada (Empresa)',
    'Concluiu o 3º EM':                         'Concluído',
    'Bolsista Universitário *Formado (a)':       'Concluído',
    'Nenhuma das opções acima':                  np.nan,
})


# ── 6. Exportar ───────────────────────────────────────────────────────────────
with pd.ExcelWriter("dados_limpos.xlsx", engine='openpyxl') as writer:
    base.to_excel(writer,              sheet_name='Base Unificada', index=False)
    selecionar(df22c).to_excel(writer, sheet_name='2022 Limpo',     index=False)
    selecionar(df23c).to_excel(writer, sheet_name='2023 Limpo',     index=False)
    selecionar(df24c).to_excel(writer, sheet_name='2024 Limpo',     index=False)

print(f"✅ dados_limpos.xlsx salvo  ({len(base)} linhas × {len(base.columns)} colunas)")


✅ dados_limpos.xlsx salvo  (3030 linhas × 21 colunas)
